In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{gold_db_name}")

df_geral = spark.read.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")


In [0]:
df_geral.printSchema()

In [0]:
# Criar a tabela fato de atendimentos
ft_atendimentos = df_geral.groupBy("id_atendente").agg(
 
    F.count("id_chamado").alias("qtd_chamados"), #qtd chamados por atendente
    
    F.avg("tempo_atendimento_segundos").alias("tempo_medio_atendimento"), #tempo médio de atendimento
    
    # percentual de casos resolvidos
    (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / 
     F.count("id_chamado") * 100).alias("taxa_resolucao"),
    
    # nota_atendimento média
    F.avg("nota_atendimento").alias("csat_medio"),
    
    # custo médio por chamado
    F.avg("valor_custo").alias("custo_medio_por_chamado"),
    
    # ADICIONAR nivel_atendimento (pegar o primeiro valor, assumindo que é fixo por atendente)
    F.first("nivel_atendimento").alias("nivel_atendimento")
)

# 2 casas decimais apenas
ft_atendimentos = ft_atendimentos.select(
    "id_atendente",
    "qtd_chamados",
    "nivel_atendimento",  # Agora existe na agregação
    F.round("tempo_medio_atendimento", 2).alias("tempo_medio_atendimento"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("custo_medio_por_chamado", 2).alias("custo_medio_por_chamado")
)

# adicionar coluna de data de processamento
ft_atendimentos = ft_atendimentos.withColumn(
    "data_processamento",
    F.current_timestamp()
)

# Ordenar por quantidade de chamados
ft_atendimentos = ft_atendimentos.orderBy(F.desc("qtd_chamados"))

# Visualizar o resultado
display(ft_atendimentos)

print("\nSchema da FT_ATENDIMENTOS:")
ft_atendimentos.printSchema()

# salvar a tabela fato
# ft_atendimentos.write.format("delta").mode("overwrite").saveAsTable(f"{catalogo}.{gold_db_name}.ft_atendimentos")

In [0]:
vw_top_performers = ft_atendimentos.filter(
    F.col("qtd_chamados") >= 100
).select(
    "id_atendente",
    "qtd_chamados",
    "nivel_atendimento",
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round(F.col("tempo_medio_atendimento") / 60, 2).alias("tempo_medio_minutos"),
    # Score composto de desempenho (peso: CSAT 40%, Taxa Resolução 40%, Velocidade 20%)
    F.round(
        (F.col("csat_medio") / 5 * 0.4) +  # Normaliza nota de 0-5 para 0-1
        (F.col("taxa_resolucao") / 100 * 0.4) +  # Normaliza % para 0-1
        (F.greatest(F.lit(0), 1 - (F.col("tempo_medio_atendimento") / 600)) * 0.2),  # Penaliza > 10min
        3
    ).alias("score_desempenho")
).orderBy(F.desc("score_desempenho"))

In [0]:
vw_top_performers.createOrReplaceTempView("vw_top_performers")

resultado = spark.sql("""
    SELECT 
        id_atendente,
        qtd_chamados,
        nivel_atendimento,
        csat_medio,
        score_desempenho
    FROM vw_top_performers
    WHERE qtd_chamados > 100
    ORDER BY score_desempenho DESC
    LIMIT 10
""")

display(resultado)

In [0]:
vw_atendentes_risco = ft_atendimentos.filter(
    (F.col("csat_medio") < 3) |  
    (F.col("taxa_resolucao") < 70) |
    (F.col("tempo_medio_atendimento") > 120)
).select(
    "id_atendente",
    "qtd_chamados",
    "nivel_atendimento",
    F.round("csat_medio", 2).alias("csat_medio"),
    F.round("taxa_resolucao", 2).alias("taxa_resolucao"),
    F.round(F.col("tempo_medio_atendimento"), 2).alias("tempo_medio_atendimento"),
    F.when(
        (F.col("csat_medio") < 3) &
        (F.col("taxa_resolucao") < 70) &
        (F.col("tempo_medio_atendimento") > 120),
        "MULTIPLOS_RISCOS"
    ).when(
        F.col("csat_medio") < 3, "CSAT_BAIXO"
    ).when(
        F.col("taxa_resolucao") < 70, "BAIXA_RESOLUCAO"
    ).when(
        F.col("tempo_medio_atendimento") > 120, "LENTO"
    ).alias("tipo_risco")

).orderBy(F.asc("csat_medio"))


In [0]:
vw_atendentes_risco.createOrReplaceTempView("vw_atendentes_risco")

resultado = spark.sql("""
    SELECT 
        id_atendente,
        qtd_chamados,
        nivel_atendimento,
        csat_medio,
        taxa_resolucao,
        tempo_medio_atendimento,
        tipo_risco,
        CASE 
            WHEN tipo_risco = 'MULTIPLOS_RISCOS' THEN 'ALTA'
            WHEN csat_medio < 2.95 AND taxa_resolucao < 65 THEN 'ALTA'
            WHEN csat_medio < 3.0 OR taxa_resolucao < 70 THEN 'MÉDIA'
            ELSE 'BAIXA'
        END AS prioridade_acao,
        -- Gap para meta
        ROUND(3 - csat_medio, 2) AS gap_csat,
        ROUND(70.0 - taxa_resolucao, 2) AS gap_taxa_resolucao
    FROM vw_atendentes_risco
    ORDER BY 
        CASE tipo_risco 
            WHEN 'MUlTIPLOS_RISCOS' THEN 1
            WHEN 'CSAT_BAIXO' THEN 2
            WHEN 'BAIXA_RESOLUCAO' THEN 3
            WHEN 'LENTO' THEN 4
        END,
        csat_medio ASC,
        qtd_chamados DESC
    LIMIT 20
""")

display(resultado)

In [0]:
vw_desempenho_por_nivel = ft_atendimentos.groupBy("nivel_atendimento").agg(
    # Quantidade de atendentes por nível
    F.count("id_atendente").alias("qtd_atendentes"),
    
    # Total de chamados por nível
    F.sum("qtd_chamados").alias("total_chamados"),
    
    # Média de chamados por atendente
    F.round(F.avg("qtd_chamados"), 2).alias("media_chamados_por_atendente"),
    
    # Média da taxa de resolução por nível
    F.round(F.avg("taxa_resolucao"), 2).alias("taxa_resolucao_media"),
    
    # Média do CSAT por nível
    F.round(F.avg("csat_medio"), 2).alias("csat_medio"),
    
    # Média do tempo de atendimento por nível (em minutos)
    F.round(F.avg("tempo_medio_atendimento") / 60, 2).alias("tempo_medio_minutos"),
    
    # Custo médio por nível
    F.round(F.avg("custo_medio_por_chamado"), 2).alias("custo_medio")
    
).orderBy("nivel_atendimento")

In [0]:
# Criar a view temporária
vw_desempenho_por_nivel.createOrReplaceTempView("vw_desempenho_por_nivel")

resultado = spark.sql("""
    SELECT 
        nivel_atendimento,
        qtd_atendentes,
        total_chamados,
        media_chamados_por_atendente,
        taxa_resolucao_media,
        csat_medio,
        tempo_medio_minutos,
        custo_medio,
        
        -- Calcular produtividade (chamados por atendente)
        ROUND(total_chamados / qtd_atendentes, 2) AS produtividade,
        
        -- Eficiência (taxa resolução / custo)
        ROUND(taxa_resolucao_media / custo_medio, 2) AS indice_eficiencia,
        
        -- Classificar o nível
        CASE 
            WHEN taxa_resolucao_media >= 75 AND csat_medio >= 3.0 THEN 'EXCELENTE'
            WHEN taxa_resolucao_media >= 70 AND csat_medio >= 2.95 THEN 'BOM'
            WHEN taxa_resolucao_media >= 60 THEN 'REGULAR'
            ELSE 'CRÍTICO'
        END AS status_nivel

    FROM vw_desempenho_por_nivel
    ORDER BY nivel_atendimento
""")

display(resultado)